# Nadie — full suite del system prompt y la memoria

Esta suite compara configuraciones candidatas con tres semillas y utiliza los escenarios como unidad estadística. Evalúa el system prompt, los dos *few-shot*, la memoria recuperada y el historial actual en el mismo orden que producción.

Ejecutala únicamente después de revisar el smoke suite. Los fallos críticos se informan por separado y nunca se compensan con un promedio alto.


## Ejecutar desde terminal

```bash
cd research/model-evaluation
source .venv/bin/activate
mkdir -p results/executed
NADIE_RUN=1 jupyter nbconvert \
  --to notebook --execute 02_full_model_evaluation.ipynb \
  --output full-executed.ipynb --output-dir results/executed \
  --ExecutePreprocessor.timeout=-1
```

Podés fijar modelo y revisión sin editar el notebook:

```bash
NADIE_RUN=1 \
NADIE_MODEL_ID=Qwen/Qwen2.5-1.5B-Instruct \
NADIE_MODEL_REVISION=main \
jupyter nbconvert --to notebook --execute 02_full_model_evaluation.ipynb \
  --output full-executed.ipynb --output-dir results/executed \
  --ExecutePreprocessor.timeout=-1
```


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd

HERE = Path.cwd().resolve()
LAB_DIR = HERE if (HERE / "evaluation_harness.py").exists() else HERE / "research/model-evaluation"
sys.path.insert(0, str(LAB_DIR))

from evaluation_harness import (
    DEFAULT_MODEL_ID,
    DEFAULT_MODEL_REVISION,
    automatic_summary,
    bootstrap_scenario_ci,
    create_blind_sheet,
    expected_run_count,
    load_experiments,
    load_scenarios,
    load_scored_results,
    prompt_snapshot,
    run_suite,
    validate_assets,
)

SUITE = "full"
MODEL_ID = os.getenv("NADIE_MODEL_ID", DEFAULT_MODEL_ID)
MODEL_REVISION = os.getenv("NADIE_MODEL_REVISION", DEFAULT_MODEL_REVISION)
RUN = os.getenv("NADIE_RUN") == "1"


## Congelar la matriz antes de ejecutar

El full suite usa todos los escenarios, tres semillas y cinco configuraciones. Las configuraciones con memoria se ejecutan solo en casos que incluyen contexto recuperado, evitando generaciones duplicadas sin memoria.


In [ ]:
assets = validate_assets()
cases = load_scenarios(SUITE)
experiments = load_experiments(SUITE)

print({
    "suite": SUITE,
    "model": MODEL_ID,
    "revision_requested": MODEL_REVISION,
    "configured_runs": expected_run_count(SUITE),
    "run_enabled": RUN,
    "prompt": prompt_snapshot(),
    "assets": assets,
})

display(pd.DataFrame(experiments)[[
    "id", "description", "scenario_filter", "memory_mode",
    "include_fewshot", "temperature", "top_p",
    "max_new_tokens", "context_window",
]])
display(pd.DataFrame({
    "id": case["id"],
    "category": case["category"],
    "memory_lines": len(case.get("memory", [])),
    "human_focus": case.get("expectations", {}).get("human_focus", ""),
} for case in cases))


## Ejecutar la matriz completa

Cada fila conserva modelo, revisión, commit, hashes del system prompt, *few-shot*, memoria, escenario y parámetros. Cambiar cualquiera de ellos genera una identidad nueva y no mezcla resultados incompatibles.


In [ ]:
if RUN:
    result_path = run_suite(SUITE, MODEL_ID, MODEL_REVISION)
    print("Results:", result_path)
else:
    print("Dry run. Set NADIE_RUN=1 to load the model and execute the suite.")


## Resumen automático

Las tasas permiten comparar configuraciones, pero no demuestran calidad humana. `critical_error_rate` es una puerta de seguridad; `behavior_failure_rate` agrega incumplimientos del comportamiento esperado por escenario.


In [ ]:
results = load_scored_results(SUITE)
if results.empty:
    print("No full results yet.")
else:
    display(automatic_summary(results).round(3))
    columns = [
        "system_hash", "experiment", "scenario_id", "seed",
        "behavior_failure", "critical_error", "unsolicited_advice",
        "memory_hit", "memory_leak", "generic_opener",
        "latency_s", "tokens_per_second", "output",
    ]
    display(results[columns].sort_values(
        ["critical_error", "behavior_failure", "experiment", "scenario_id", "seed"],
        ascending=[False, False, True, True, True],
    ))


## Intervalos de confianza por escenario

Las semillas son repeticiones del mismo caso, no observaciones independientes. Por eso primero se promedian dentro de cada escenario y después se remuestrean escenarios completos.


In [ ]:
if not results.empty:
    ci_frames = [
        bootstrap_scenario_ci(results, "behavior_failure"),
        bootstrap_scenario_ci(results, "critical_error"),
        bootstrap_scenario_ci(results, "unsolicited_advice"),
        bootstrap_scenario_ci(results, "memory_hit"),
    ]
    confidence_intervals = pd.concat(
        [frame for frame in ci_frames if not frame.empty],
        ignore_index=True,
    )
    display(confidence_intervals.round(3))


## Comparación pareada de la memoria

Esta tabla usa únicamente escenarios con memoria y semillas presentes en ambas configuraciones. Un valor negativo en `failure_difference` favorece a la configuración con memoria.


In [ ]:
if not results.empty:
    paired_source = results[results["experiment"].isin([
        "system_fewshot", "system_fewshot_memory"
    ])].copy()
    paired_source = paired_source[paired_source["memory_context"].map(bool) | paired_source["scenario_id"].isin(
        set(results.loc[results["memory_mode"] == "scenario", "scenario_id"])
    )]
    paired = paired_source.pivot_table(
        index=["scenario_id", "seed"],
        columns="experiment",
        values="behavior_failure",
        aggfunc="first",
    ).dropna()
    if paired.empty:
        print("No paired memory comparison available yet.")
    else:
        paired["failure_difference"] = (
            paired["system_fewshot_memory"] - paired["system_fewshot"]
        )
        display(paired.reset_index())
        print("Mean paired difference:", round(paired["failure_difference"].mean(), 3))


## Evaluación humana ciega

La hoja incluye conversación y memoria para que sea posible juzgar pertinencia, pero oculta modelo, prompt y configuración. Deben revisarla al menos dos personas por separado antes de abrir el mapa.


In [ ]:
if not results.empty:
    blind_path, map_path = create_blind_sheet(results, SUITE)
    print("Blind review sheet:", blind_path)
    print("Mapping — keep closed until reviews finish:", map_path)


## Puerta de decisión

Un candidato no avanza solamente porque su promedio sea mejor. Debe cumplir:

1. cero fallos críticos en el conjunto disponible;
2. ninguna regresión clara al agregar memoria;
3. menor consejo no solicitado sin perder utilidad cuando se pide ayuda;
4. evaluación humana ciega favorable;
5. repetición posterior con el artefacto `q4f16_1-MLC` en WebLLM y dispositivos reales.

Esta primera versión tiene 20 escenarios. Sirve para comparación y regresión, pero todavía no reemplaza el banco bloqueado de seguridad propuesto en `../safety-evaluation-plan.md`.
